In [4]:
# Model Training

### Libraries
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
import pandas as pd




In [5]:
train_data = pd.read_csv("../data/train_data.csv")
test_data = pd.read_csv("../data/test_data.csv")




In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode_texts(texts, tokenizer):
    return tokenizer(texts.tolist(), padding=True, truncation=True, return_tensors="tf")

train_encodings = encode_texts(train_data["text"], tokenizer)
test_encodings = encode_texts(test_data["text"], tokenizer)




In [7]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_data["label"].tolist()
))

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_data["label"].tolist()
))



In [8]:
model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

# Using test_dataset as validation data due to small dataset size for now
model.fit(train_dataset.shuffle(1000).batch(2), epochs=3, validation_data=test_dataset.batch(2))
model.save_pretrained("../models")




model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3


1/1 [==============================] - 41s 41s/step - loss: 0.6534 - accuracy: 1.0000 - val_loss: 0.7803 - val_accuracy: 0.0000e+00
Epoch 2/3
1/1 [==============================] - 1s 1s/step - loss: 0.5643 - accuracy: 1.0000 - val_loss: 0.8281 - val_accuracy: 0.0000e+00
Epoch 3/3
1/1 [==============================] - 1s 880ms/step - loss: 0.5549 - accuracy: 1.0000 - val_loss: 0.8890 - val_accuracy: 0.0000e+00
